In [ ]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score


# Caricamento dataset
def f(i):
    return pd.read_csv(i, 
                       usecols=["ts","src_bytes","missed_bytes","label","type","conn_state", "duration"], 
                       na_values="-",
                       #skiprows=lambda i: i > 0 and random.random() > 0.125
                       )

#dataset_path = Path(f"assets/Processed_Network_dataset/Network_dataset_3.csv")
#df = pd.read_csv(dataset_path, 
#                 usecols=["ts","src_bytes","missed_bytes","label","type","conn_state", "duration"],
#                 na_values="-")

df = pd.concat(map(f, [ 
                       'assets/Processed_Network_dataset/Network_dataset_2.csv',
                       'assets/Processed_Network_dataset/Network_dataset_3.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_4.csv',
                       'assets/Processed_Network_dataset/Network_dataset_5.csv',
                       'assets/Processed_Network_dataset/Network_dataset_6.csv',
                       'assets/Processed_Network_dataset/Network_dataset_7.csv',
                       'assets/Processed_Network_dataset/Network_dataset_8.csv',
                       'assets/Processed_Network_dataset/Network_dataset_9.csv',
                       'assets/Processed_Network_dataset/Network_dataset_10.csv',
                       'assets/Processed_Network_dataset/Network_dataset_11.csv',
                       'assets/Processed_Network_dataset/Network_dataset_12.csv',
                       'assets/Processed_Network_dataset/Network_dataset_13.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_14.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_15.csv',
                       'assets/Processed_Network_dataset/Network_dataset_16.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_17.csv',
                       'assets/Processed_Network_dataset/Network_dataset_18.csv',
                       'assets/Processed_Network_dataset/Network_dataset_19.csv',
                       'assets/Processed_Network_dataset/Network_dataset_20.csv',
                       'assets/Processed_Network_dataset/Network_dataset_21.csv']))
df = df.dropna()

"""target_per_class = 50000
df = (
    df.groupby('type', group_keys=False)
      .apply(lambda x: x.sample(
          n=min(len(x), target_per_class),
          random_state=42))
      .reset_index(drop=True)
)"""

print("Count: ", df['type'].value_counts())


In [ ]:
target_col = df.columns[-1]
y = df[target_col]
df = df.drop(columns=[target_col])

#separazione colonne
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
boolean_cols = [col for col in df.columns if df[col].dropna().nunique() == 2 and df[col].dropna().isin([0, 1]).all()]
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.difference(boolean_cols).tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
cols_to_scale = list(set(numeric_cols) & set(df_encoded.columns))
df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])
x = df_encoded.astype('float32').values
print(x.shape) 

le = LabelEncoder()
y_encoded = le.fit_transform(y)  # ['normal', 'dos', ...] → [0, 1, ...]


x_tensor = torch.tensor(x, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

print("X: ", x)
print("Y encoded: ", y_encoded)

# Split train/test
x_train, x_test, y_train, y_test = train_test_split(
    x_tensor, y_tensor, test_size=0.2, random_state=42
)

class NetworkDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

batch_size = 64
train_loader = DataLoader(NetworkDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(NetworkDataset(x_test, y_test), batch_size=batch_size)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

#Creazione modello MLP

class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.layers(x)

num_classes = len(le.classes_)
print("Num class: ", num_classes)
model = MLP(input_dim=x.shape[1], num_classes=num_classes)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#training loop

epochs = 10
loss_values=[]
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        #print("Outputs: ", outputs)
        loss = criterion(outputs, labels)
        #print("Inner Loss: ", loss)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_list = epoch_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss_list:.4f}")
    loss_values.append(loss_list)


model.eval()
y_pred_list = []

with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        y_pred_list.append(preds)

y_pred = torch.cat(y_pred_list).numpy()
y_true = y_test.numpy()

acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {acc:.4f}")

plt.plot(loss_values)
print(loss_values)

# Report con precision, recall, f1-score per ogni classe
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import torch
#from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from utils.data import train_val_test_split

def dirichlet_split(y, n_clients, alpha):
    num_c = len(np.unique(y))
    net_dataidx_map = {}
    p_client = np.zeros((n_clients, num_c))
    for i in range(n_clients):
        p_client[i] = np.random.dirichlet(np.repeat(alpha, num_c))
    idx_batch = [[] for _ in range(n_clients)]
    for k in range(num_c):
        idx_k = np.where(y == k)[0]
        np.random.shuffle(idx_k)
        proportions = p_client[:, k]
        proportions = proportions / proportions.sum()
        split_points = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
        idx_batch = [
            idx_j + idx.tolist()
            for idx_j, idx in zip(idx_batch, np.split(idx_k, split_points))
        ]
    for j in range(n_clients):
        np.random.shuffle(idx_batch[j])
        net_dataidx_map[j] = idx_batch[j]
    net_cls_counts = {}
    for net_i, dataidx in net_dataidx_map.items():
        unq, unq_cnt = np.unique(y.iloc[dataidx], return_counts=True)
        net_cls_counts[net_i] = dict(zip(unq, unq_cnt))
    return net_dataidx_map, net_cls_counts


#Y_train_prova = next(iter(train_loader))[1].numpy()
#inds, net_cls_counts = dirichlet_split(Y_train_prova, 10, 0.05, 10)

#print("Inds", inds)
#print("Net cls count", net_cls_counts)

n_nodes = 10
k=3
n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

df_full = df_encoded.copy()
df_full[target_col] = y_encoded
df_full = df_full.sample(frac=1, random_state=np.random.RandomState(20)).reset_index(drop=True)
#splits = np.array_split(df_full, n_nodes)
#splits = np.array_split(df_full, 10)

#for i in range(n_simulations):
i = 0
split_df = df_full
dataset_folder = Path(output_path / str(i) / "4inTestBalancedAlpha02_seed20")
dataset_folder.mkdir(parents=True, exist_ok=True)

target_col = split_df.columns[-1]
y = split_df[target_col]
x = split_df.drop(columns=[target_col])

node_map, _ = dirichlet_split(y, n_clients=n_nodes, alpha=0.2)

for node_id, idxs in node_map.items():
    X_node = x.iloc[idxs]
    Y_node = y.iloc[idxs]

    X_train, Y_train, X_val, Y_val, X_test, Y_test = train_val_test_split(
            X_node, Y_node, test_perc=0.3, val_perc_on_train=0.1
        )

    np.savez(
            dataset_folder / f"node_{node_id}.npz",
            X_train=X_train.to_numpy(),
            Y_train=Y_train.to_numpy(),
            X_val=X_val.to_numpy(),
            Y_val=Y_val.to_numpy(),
            X_test=X_test.to_numpy(),
            Y_test=Y_test.to_numpy(),
        )
    #split_df.to_parquet(dataset_folder / f"node_{node_id}.parquet", index=False)
    print(f"Node {node_id} in simulation {i} saved!")
print(f"Simulation {i} saved!")


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from utils.data import train_val_test_split

n_nodes = 2
k=1
n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

df_full = df_encoded.copy()
df_full[target_col] = y_encoded

#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

#splits = []
#for _, idx in skf.split(df_full.drop(columns=[target_col]), y_encoded):
#    split_df = df.iloc[idx].reset_index(drop=True)
#    splits.append(split_df)



#df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)
splits = np.array_split(df_full, n_nodes)

i = 0
#for i in range(n_simulations):
for j, split_df in enumerate(splits):
        dataset_folder = Path(output_path / str(i) / "4inTestSplit")
        dataset_folder.mkdir(parents=True, exist_ok=True)
        target_col = split_df.columns[-1]
        y = split_df[target_col]
        x = split_df.drop(columns=[target_col])
        X_train, Y_train, X_val, Y_val, X_test, Y_test = train_val_test_split(x, y, test_perc=0.3, val_perc_on_train=0.1)
        np.savez(
            Path(dataset_folder / f"node_{j}"), 
            X_train=X_train,
            Y_train=Y_train,
            X_val=X_val,
            Y_val=Y_val,
            X_test=X_test,
            Y_test=Y_test)
        #split_df.to_parquet(dataset_folder / f"node_{j}.parquet", index=False)
        print(f"Node {j} in simulation {i} saved!")
print(f"Simulation {i} saved!")


In [ ]:
"""import numpy as np


n_nodes = 10
k=3
n_simulations = 10
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

df_full = df_encoded.copy()
df_full[target_col] = y_encoded

splits = np.array_split(df_full, n_nodes)

for i in range(n_simulations):
    for j, split_df in enumerate(splits):
        dataset_folder = Path(output_path / str(i) )
        dataset_folder.mkdir(parents=True, exist_ok=True)
        split_df.to_parquet(dataset_folder / f"node_{j}.parquet", index=False)
        print(f"Node {j} in simulation {i} saved!")
    print(f"Simulation {i} saved!")"""


In [ ]:
"""from utils.data import prepare_dataset_for_training

ds_name = "4in"

for n in range(n_simulations):
    nodes_datasets = [
        pd.read_parquet(output_path / str(n) / f"node_{i}.parquet") for i in range(n_nodes)
    ]
    
    start_i = round(len(nodes_datasets[0]) * 0.8)

    nodes_datasets = [
        ts
        for i, ts in enumerate(nodes_datasets)
    ]
    
    prepare_dataset_for_training(
        towers_datasets=nodes_datasets,
        output_folder=output_path / str(n) / ds_name,
        input_timesteps=4,
        output_timesteps=1,
        n_functions=1,
        n_auxiliary_features=8,
    )"""

In [ ]:
#print("test: ", nodes_datasets)

### Simulation 

In [ ]:

import functools
import json
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary

from gossiplearning.config import Config
from utils.gossip_training import get_node_dataset, round_trip_fn, model_transmission_fn, \
    run_simulation
from utils.model_creators import create_MLP
from utils.multiprocessing_test import run_in_parallel

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf

n_nodes = 10
k=3
n_simulations = 10
#n_nodes = 2
#k=1
#n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")


In [ ]:
with open("TON_Iot_config.json", "r") as f:
#with open("2_NODES_config.json", "r") as f:
    config = json.load(f)

config = Config.model_validate(config)

workspace_path = Path(config.workspace_dir)
workspace_path.mkdir(exist_ok=True, parents=True)
with (workspace_path / "config.json").open("w") as f:
    json.dump(config.model_dump(), f, indent=True)

In [ ]:
#print(x.shape)  # dovrebbe stampare qualcosa tipo (N, 17)

#summary(model, input_size=(x.shape[1], ))

#model_creator = functools.partial(create_MLP, config=config)

#summary(model_creator(), input_size=(x.shape[1], ))

In [ ]:
"""from utils.data import get_common_test_set
from gossiplearning.weight import weight_by_dataset_size

model_creator = functools.partial(create_MLP, config=config)
i = 1

for i in range(2):
     run_simulation(
           config=config,
           simulation_number=i,
           network_folder=network_path / str(i),
           round_trip_fn=round_trip_fn,
           model_transmission_fn=model_transmission_fn,
           node_data_fn=functools.partial(
               get_node_dataset, 
               base_folder=output_path, 
               simulation_number=i, 
               ds_name="4inTest"
           ),
           model_creator=model_creator,
           get_test_set = functools.partial(
                get_common_test_set,
                node_data_fn=functools.partial(
                                get_node_dataset,
                                base_folder=output_path, 
                                simulation_number=i, 
                                ds_name="4inTest"
                            ),
                n_nodes=config.n_nodes,
                perc=0.1
            ),
           weight_fn=weight_by_dataset_size
        )"""

In [ ]:
#SINGLE SIM
"""from utils.data import get_common_test_set
from gossiplearning.weight import weight_by_dataset_size

model_creator = functools.partial(create_MLP, config=config)
i = 0

run_simulation(
           config=config,
           simulation_number=i,
           network_folder=network_path / str(i),
           round_trip_fn=round_trip_fn,
           model_transmission_fn=model_transmission_fn,
           node_data_fn=functools.partial(
               get_node_dataset, 
               base_folder=output_path, 
               simulation_number=i, 
               ds_name="4inTestBalancedAlpha02_seed"
           ),
           model_creator=model_creator,
           get_test_set = functools.partial(
                get_common_test_set,
                node_data_fn=functools.partial(
                                get_node_dataset,
                                base_folder=output_path, 
                                simulation_number=i, 
                                ds_name="4inTestBalancedAlpha02_seed"
                            ),
               n_nodes=config.n_nodes,
                perc=0.1
            ),
          weight_fn=weight_by_dataset_size
        )"""

In [ ]:
from utils.data import get_common_test_set
from gossiplearning.weight import weight_by_dataset_size

model_creator = functools.partial(create_MLP, config=config)
#i = 5


run_in_parallel(
    [
        functools.partial(
           run_simulation,
           config=config,
           simulation_number=i,
           network_folder=network_path / str(i),
           round_trip_fn=round_trip_fn,
           model_transmission_fn=model_transmission_fn,
           node_data_fn=functools.partial(
               get_node_dataset, 
               base_folder=output_path, 
               simulation_number=i, 
               ds_name="4inTestBalancedAlpha02_seed"
           ),
           model_creator=model_creator,
           get_test_set = functools.partial(
                get_common_test_set,
                node_data_fn=functools.partial(
                                get_node_dataset,
                                base_folder=output_path, 
                                simulation_number=i, 
                                ds_name="4inTestBalancedAlpha02_seed"
                            ),
                n_nodes=config.n_nodes,
                perc=0.1
            ),
           weight_fn=weight_by_dataset_size
        )
        for i in range(2)
    ]
)